<a href="https://colab.research.google.com/github/Lattemelia/Portfolio/blob/main/%E3%80%900624%EF%BD%9E%E7%8F%BE%E5%9C%A8%E4%BD%9C%E6%88%90%E4%B8%AD%E3%80%91/%E4%B8%AD%E5%8F%A4%E8%BB%8A%E3%81%AE%E4%BE%A1%E6%A0%BC%E4%BA%88%E6%B8%AC(KaggleConpe).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 新しいセクション

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [28]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [29]:
PATH = r"/content/drive/MyDrive/Colab Notebooks"

test_df = pd.read_csv(PATH + r"/test.csv")
train_df = pd.read_csv(PATH + r"/train.csv")

marged_df = pd.concat([test_df,train_df],axis=0)

In [44]:
marged_df.isnull().sum()

,0
id,0
brand,0
model,0
model_year,0
milage,0
fuel_type,8466
engine,0
transmission,0
ext_col,0
int_col,0


In [31]:
cols_to_lower = ['brand', 'model', 'engine', 'fuel_type', 'transmission']

for col in cols_to_lower:
    marged_df[col] = marged_df[col].str.lower().str.strip()

In [32]:
# (?i) を先頭に配置することで、パターン全体の大文字小文字を無視します
hp_pattern = r"(?i)(\d+\.\d+|\d+)\s*hp"

# engine列の馬力表記を丸ごと削除
marged_df["engine"] = (
    marged_df["engine"]
    .astype(str)
    .str.replace(hp_pattern, "", regex=True)
    .str.strip()
)

# 連続したスペースを1つに統合
marged_df["engine"] = marged_df["engine"].str.replace(r"\s+", " ", regex=True)

In [33]:
import pandas as pd


def extract_model_by_word_count(model_str):
    if pd.isna(model_str):
        return None

    # 文字列を前後の空白を消して、スペースで分割（単語のリストにする）
    words = str(model_str).strip().split()
    word_count = len(words)

    if word_count == 0:
        return None
    # 1単語のみの場合（例: "Passat"）は、そのままその単語を返す
    elif word_count == 1:
        return words[0]
    # 2単語の場合（例: "A3 2.0T" ➔ "A3 2.0T"）は、そのまま2単語結合して返す
    elif word_count == 2:
        return " ".join(words)
    # 3単語以上の場合（例: "Golf Alltrack TSI SE" ➔ "Golf Alltrack"）は、最初の2単語を返す
    else:
        return " ".join(words[:2])


# ─── marged_df への適用 ───
# model列にルールを適用し、新しく 'pure_model' カラムを作成します
marged_df["pure_model"] = marged_df["model"].apply(extract_model_by_word_count)

marged_df["pure_model"].value_counts()

,count
pure_model,
rover range,12795
911 carrera,6457
f-150 xlt,5353
e-class e,5204
corvette stingray,4104
m3 base,3641
mustang gt,3221
gx 460,3053
c-class c,2950


In [34]:
import re
import numpy as np
import pandas as pd

# ─── 1. 排気量を抽出するための正規表現パターン ───
# (?i) : パターンの先頭に置き、大文字小文字をすべて無視（L, l, liter, Litter, liters に対応）
# (\d+\.\d+|\d+) : グループ1。排気量の数値（3.0 や 2 など）をキャプチャ
# \s* : 数値と単位の間にあるかもしれない空白（0文字以上）
# (l|liter|liters|litres) : 単位の揺れをカバー
disp_pattern = r"(?i)(\d+\.\d+|\d+)\s*(l|liter|liters|litres)"


# ─── 2. 抽出用の関数を定義 ───
def extract_displacement(engine_text):
    if pd.isna(engine_text):
        return np.nan

    match = re.search(disp_pattern, str(engine_text))
    if match:
        try:
            # マッチした最初のグループ（数値部分）をfloat型に変換して返す
            return float(match.group(1))
        except ValueError:
            return np.nan
    return np.nan


# ─── 3. marged_df への適用（抽出） ───
# engine列から排気量を抜き出し、新列 'displacement_l' を作成
marged_df["displacement_l"] = marged_df["engine"].apply(extract_displacement)

#排気量以外の情報を'engine_type'へ入力
marged_df["engine_type"] = (
    marged_df["engine"]
    .astype(str)
    .str.replace(disp_pattern, "", regex=True)
    .str.strip()
)

# 文字を消した後に発生する「2連スペース」を1つのスペースに美しく統合
marged_df["engine"] = marged_df["engine"].str.replace(r"\s+", " ", regex=True)

In [45]:
import numpy as np
import pandas as pd


def advanced_engine_classifier_v4(engine_str):
    # 最初から欠損しているものは None, NaN を返す
    if pd.isna(engine_str):
        return pd.Series([None, np.nan])

    s = str(engine_str).lower().strip()

    # ─── ① 【修正】電気自動車（EV）の判定 ───
    # 気筒数も排気量も「存在しない」ため、どちらも欠損値（None, np.nan）を返す
    if ("electric" in s or "battery" in s or "dual ac" in s) and not any(
        x in s for x in ["hybrid", "plug-in", "gas", "mild"]
    ):
        return pd.Series([None, np.nan])

    # ─── ② 気筒数キーワードによる判定 ───
    if any(x in s for x in ["12 cylinder", "v12", "w12", "w16"]):
        return pd.Series(["V12/W16", 6.0])

    if any(x in s for x in ["10 cylinder", "v10"]):
        return pd.Series(["V10", 5.2])

    if any(x in s for x in ["8 cylinder", "v8"]):
        return pd.Series(["V8", 4.0])

    if any(
        x in s
        for x in ["6 cylinder", "v6", "straight 6", "flat 6", "i6", "h6"]
    ):
        return pd.Series(["V6", 3.0])

    if any(x in s for x in ["5 cylinder", "i5"]):
        return pd.Series(["I5", 2.5])

    if (
        any(x in s for x in ["4 cylinder", "i4", "h4", "tsi", "tfsi", "gdi"])
        or "i-vtec v6" not in s
        and "i4" in s
    ):
        return pd.Series(["I4", 2.0])

    if any(x in s for x in ["3 cylinder", "i3"]):
        return pd.Series(["I3", 1.5])

    if "rotary" in s:
        return pd.Series(["Rotary", 1.3])

    if "hybrid" in s or "plug-in" in s:
        return pd.Series(["I4", 2.0])

    # ─── ③ 分からないものも None, np.nan を返す ───
    return pd.Series([None, np.nan])


# marged_dfへの一括適用
marged_df[["engine_type_clean", "displacement_l"]] = marged_df["engine"].apply(
    advanced_engine_classifier_v4
)

In [42]:
marged_df[["engine_type_clean", "displacement_l"]].value_counts()

,,count
engine_type_clean,displacement_l,
V6,3.0,126244
V8,4.0,98927
I4,2.0,60447
V12/W16,6.0,3099
V10,5.2,1803
I5,2.5,1082
I3,1.5,439
Rotary,1.3,105
